<a href="https://colab.research.google.com/github/asteriaji/MSSP6070/blob/main/WeeklyModules/Week05/Week_05_Data_Wrangling_Join_Combine_Improved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 05 — Data Wrangling: Join, Combine, and Reshape

This lesson introduces several of the most important pandas techniques for reorganizing and integrating data:

- hierarchical indexing (`MultiIndex`)
- database-style joins with `merge()`
- index-based joins with `join()`
- row/column concatenation with `concat()`
- filling overlapping data with `combine_first()`
- reshaping with `stack()`, `unstack()`, `pivot()`, and `melt()`

**Learning objectives**

By the end of the lesson, students should be able to:
1. Explain the difference between a key, an index, and a hierarchical index.
2. Select the correct join type (`inner`, `left`, `right`, `outer`) for a business question.
3. Detect and explain row multiplication caused by duplicate join keys.
4. Combine datasets vertically and horizontally with `pd.concat()`.
5. Reshape data between long and wide formats.
6. Validate a wrangling operation by checking row counts, missing values, and key uniqueness.

Content adapted from Wes McKinney, *Python for Data Analysis*, and the pandas documentation.


In [ ]:
# Optional Google Colab setup.
# This cell runs only when the notebook is actually being executed in Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('Not running in Google Colab; skipping Google Drive mount.')


In [ ]:
# Optional course-specific connection/setup script.
# Keeping environment-specific configuration optional makes the lesson portable.
from pathlib import Path

connection_script = Path('/content/drive/MyDrive/Google_connection.py')
if IN_COLAB and connection_script.exists():
    get_ipython().run_line_magic('run', str(connection_script))
else:
    print('Course connection script not found or not required for this lesson.')


In [ ]:
# Set a course working directory only when it exists.
from pathlib import Path

course_dir = Path('/content/MSSP-607/WeeklyModules/Week05')
if course_dir.exists():
    %pushd /content/MSSP-607/WeeklyModules/Week05
else:
    print(f'Course directory not found: {course_dir}')
    print('Continuing from the current working directory.')


In [ ]:
# Core libraries used in this lesson.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.options.display.max_rows = 20
np.random.seed(12345)  # Reproducible demonstration data.
plt.rc('figure', figsize=(10, 6))
np.set_printoptions(precision=4, suppress=True)

print('pandas version:', pd.__version__)


## 1. Hierarchical Indexing (`MultiIndex`)

A standard pandas object has one label per axis position. A **hierarchical index** stores multiple index levels for each row or column. This is useful when one observation is naturally identified by more than one dimension—for example, *state + year* or *customer + order*.

**Teaching point:** A `MultiIndex` is not the same as having several ordinary columns. It changes how pandas labels and selects the data.


In [ ]:
#Create two indexes on multiple levels: letters then numbers.
data = pd.Series(np.random.randn(9),
                 index=[['a', 'a', 'a', 'b', 'b', 'c', 'c', 'd', 'd'],
                        [1, 2, 3, 1, 3, 1, 2, 2, 3]])
data

In [ ]:
data.index  #List the tuples that form the indexes.

In [ ]:
# Different ways to select from the first index level.
display(data['b'])
display(data['b':'c'])
display(data.loc[['b', 'd']])


In [ ]:
# Select all rows whose second index level equals 2.
# The tuple syntax makes the two index dimensions explicit.
data.loc[(slice(None), 2)]


In [ ]:
#Produce a Pandas Dataframe from the indexes.
data.unstack()

In [ ]:
#Unstack then restack the indexes.
data.unstack().stack()

In [ ]:
#In NumPy, np.arange() is a function used to create an array with regularly
#spaced values within a specified interval. It returns an array containing
#numbers that start from a specified value and go up to, but not including,
#an endpoint value.
frame = pd.DataFrame(np.arange(12).reshape((4, 3)),
                     index=[['a', 'a', 'b', 'b'], [1, 2, 1, 2]],
                     columns=[['Ohio', 'Ohio', 'Colorado'],
                              ['Green', 'Red', 'Green']])
frame

In [ ]:
#Rename indexes and columns of the dataframe for each index level.
frame.index.names = ['key1', 'key2']
frame.columns.names = ['state', 'color']
frame

In [ ]:
frame['Ohio']

A `MultiIndex` can also be built explicitly. The equivalent construction for the column labels above is:

```python
pd.MultiIndex.from_arrays(
    [['Ohio', 'Ohio', 'Colorado'], ['Green', 'Red', 'Green']],
    names=['state', 'color']
)
```

This explicit form is useful when index levels are being assembled programmatically.


### Reordering and Sorting Levels

In [ ]:
frame.swaplevel('key1', 'key2')

In [ ]:
# Reordering a MultiIndex and then sorting by a level.
display(frame.sort_index(level='key2'))
display(frame.swaplevel('key1', 'key2').sort_index(level='key2'))


### Summary Statistics by Level

In [ ]:
# Aggregation can be performed across columns or rows.
print('Row totals:')
display(frame.sum(axis=1))
print('Column totals:')
display(frame.sum(axis=0))


### Indexing with a DataFrame's columns

In [ ]:
frame = pd.DataFrame({'a': range(7), 'b': range(7, 0, -1),
                      'c': ['one', 'one', 'one', 'two', 'two',
                            'two', 'two'],
                      'd': [0, 1, 2, 0, 1, 2, 3]})
frame

In [ ]:
frame2 = frame.set_index(['c', 'd'])
frame2

In [ ]:
frame.set_index(['c', 'd'], drop=False)

In [ ]:
frame2.reset_index()

### Instructor note: choosing the right combining operation

Before demonstrating `merge()`, ask students: **“Are we matching records because they share a key, or are we simply stacking compatible data?”** This distinction separates `merge()` from `concat()` and prevents many early pandas mistakes.


## Combining and Merging Datasets

### Database-Style DataFrame Joins

`pd.merge()` combines rows by matching one or more key columns, similar to a SQL `JOIN`.

**Four questions to ask before every merge:**
1. Which column(s) form the key?
2. Are the keys unique in the left table, the right table, both, or neither?
3. Which rows should survive when a match is absent?
4. How will we validate the result?

A merge can silently create **more rows than either input** when duplicate keys exist. This is expected behavior for many-to-many joins, but it should never be a surprise.


In [ ]:
df1 = pd.DataFrame({'key': ['b', 'b', 'a', 'c', 'a', 'a', 'b'],
                    'data1': range(7)})
df2 = pd.DataFrame({'key': ['a', 'b', 'd'],
                    'data2': range(3)})

print('Left table:')
display(df1)
print('Right table:')
display(df2)


In [ ]:
# If the common key column has the same name, pandas can infer it.
merged_default = pd.merge(df1, df2)
merged_default


In [ ]:
# Explicit is usually better for teaching and production code.
merged_on_key = pd.merge(df1, df2, on='key', validate='many_to_one')
merged_on_key


In [ ]:
df3 = pd.DataFrame({'lkey': ['b', 'b', 'a', 'c', 'a', 'a', 'b'],
                    'data1': range(7)})
df4 = pd.DataFrame({'rkey': ['a', 'b', 'd'],
                    'data2': range(3)})
pd.merge(df3, df4, left_on='lkey', right_on='rkey')

In [ ]:
# Outer join: preserve every key from both tables.
outer_merge = pd.merge(df1, df2, on='key', how='outer', indicator=True)
outer_merge


In [ ]:
# Demonstrate a many-to-many merge.
df1 = pd.DataFrame({'key': ['b', 'b', 'a', 'c', 'a', 'b'],
                    'data1': range(6)})
df2 = pd.DataFrame({'key': ['a', 'b', 'a', 'b', 'd'],
                    'data2': range(5)})

print('Left key counts:')
display(df1['key'].value_counts())
print('Right key counts:')
display(df2['key'].value_counts())

many_to_many = pd.merge(df1, df2, on='key', how='left', validate='many_to_many')
print(f'Rows before merge: left={len(df1)}, right={len(df2)}')
print(f'Rows after merge: {len(many_to_many)}')
many_to_many


In [ ]:
# Inner join keeps only keys that appear in both tables.
inner_merge = pd.merge(df1, df2, on='key', how='inner')
inner_merge


In [ ]:
left = pd.DataFrame({'key1': ['foo', 'foo', 'bar'],
                     'key2': ['one', 'two', 'one'],
                     'lval': [1, 2, 3]})
right = pd.DataFrame({'key1': ['foo', 'foo', 'bar', 'bar'],
                      'key2': ['one', 'one', 'one', 'two'],
                      'rval': [4, 5, 6, 7]})
pd.merge(left, right, on=['key1', 'key2'], how='outer')

In [ ]:
# If non-key column names overlap, suffixes keep their origins clear.
print('Merge on key1 only:')
display(pd.merge(left, right, on='key1'))

print('Same merge with explicit suffixes:')
display(pd.merge(left, right, on='key1', suffixes=('_left', '_right')))


### Instructor note: merge validation

Use `validate=` whenever you know the intended relationship:

- `one_to_one`
- `one_to_many`
- `many_to_one`
- `many_to_many`

This turns assumptions about key uniqueness into executable checks. For production data pipelines, that is much safer than visually inspecting the result.


### Merging on Index

In [ ]:
left1 = pd.DataFrame({'key': ['a', 'b', 'a', 'a', 'b', 'c'],
                      'value': range(6)})
right1 = pd.DataFrame({'group_val': [3.5, 7]}, index=['a', 'b'])

display(left1)
display(right1)

pd.merge(left1, right1, left_on='key', right_index=True, validate='many_to_one')


In [ ]:
pd.merge(left1, right1, left_on='key', right_index=True, how='outer')

In [ ]:
lefth = pd.DataFrame({'key1': ['Ohio', 'Ohio', 'Ohio', 'Nevada', 'Nevada'],
                      'key2': [2000, 2001, 2002, 2001, 2002],
                      'data': np.arange(5.)})
righth = pd.DataFrame(
    np.arange(12).reshape((6, 2)),
    index=pd.MultiIndex.from_arrays(
        [['Nevada', 'Nevada', 'Ohio', 'Ohio', 'Ohio', 'Ohio'],
         [2001, 2000, 2000, 2000, 2001, 2002]],
        names=['key1', 'key2']
    ),
    columns=['event1', 'event2']
)

display(lefth)
display(righth)


In [ ]:
print('Inner merge on two left-side columns and a MultiIndex on the right:')
display(pd.merge(lefth, righth,
                 left_on=['key1', 'key2'], right_index=True))

print('Outer merge:')
display(pd.merge(lefth, righth,
                 left_on=['key1', 'key2'], right_index=True, how='outer'))


In [ ]:
left2 = pd.DataFrame([[1., 2.], [3., 4.], [5., 6.]],
                     index=['a', 'c', 'e'],
                     columns=['Ohio', 'Nevada'])
right2 = pd.DataFrame([[7., 8.], [9., 10.], [11., 12.], [13, 14]],
                      index=['b', 'c', 'd', 'e'],
                      columns=['Missouri', 'Alabama'])

display(left2)
display(right2)
pd.merge(left2, right2, how='outer', left_index=True, right_index=True)


In [ ]:
left2.join(right2, how='outer')

In [ ]:
left1.join(right1, on='key')

In [ ]:
another = pd.DataFrame([[7., 8.], [9., 10.], [11., 12.], [16., 17.]],
                       index=['a', 'c', 'e', 'f'],
                       columns=['New York', 'Oregon'])

display(another)
print('Default index join:')
display(left2.join([right2, another]))
print('Outer index join:')
display(left2.join([right2, another], how='outer'))


### Concatenating Along an Axis

Use `pd.concat()` when you want to **stack compatible objects**, not match rows by a relational key.

- `axis=0` stacks rows vertically.
- `axis=1` places objects side-by-side and aligns by index.
- `join='outer'` keeps the union of labels.
- `join='inner'` keeps only labels shared by all objects.
- `ignore_index=True` creates a fresh integer index.
- `keys=` adds a new hierarchical level that records each object's source.

**Rule of thumb:** `merge()` matches records by keys; `concat()` appends/aligned objects along an axis.


In [ ]:
arr = np.arange(12).reshape((3, 4))
arr
np.concatenate([arr, arr], axis=1)

In [ ]:
s1 = pd.Series([0, 1], index=['a', 'b'])
s2 = pd.Series([2, 3, 4], index=['c', 'd', 'e'])
s3 = pd.Series([5, 6], index=['f', 'g'])

In [ ]:
pd.concat([s1, s2, s3])

In [ ]:
pd.concat([s1, s2, s3], axis=1)

In [ ]:
s4 = pd.concat([s1, s3])
print('Combined Series s4:')
display(s4)

print('Outer alignment along columns:')
display(pd.concat([s1, s4], axis=1))

print('Inner alignment along columns:')
display(pd.concat([s1, s4], axis=1, join='inner'))


In [ ]:
#Concatenate the two dataframes and reindex with new attributes.
pd.concat([s1, s4], axis=1).reindex(['a', 'c', 'b', 'e'])

In [ ]:
result = pd.concat([s1, s1, s3], keys=['one', 'two', 'three'])
result
result.unstack()

In [ ]:
pd.concat([s1, s2, s3], axis=1, keys=['one', 'two', 'three'])

In [ ]:
df1 = pd.DataFrame(np.arange(6).reshape(3, 2), index=['a', 'b', 'c'],
                   columns=['one', 'two'])
df2 = pd.DataFrame(5 + np.arange(4).reshape(2, 2), index=['a', 'c'],
                   columns=['three', 'four'])

display(df1)
display(df2)
pd.concat([df1, df2], axis=1, keys=['level1', 'level2'])


In [ ]:
pd.concat({'level1': df1, 'level2': df2}, axis=1)

In [ ]:
pd.concat([df1, df2], axis=1, keys=['level1', 'level2'],
          names=['upper', 'lower'])

In [ ]:
df1 = pd.DataFrame(np.random.randn(3, 4), columns=['a', 'b', 'c', 'd'])
df2 = pd.DataFrame(np.random.randn(2, 3), columns=['b', 'd', 'a'])

display(df1)
display(df2)


In [ ]:
pd.concat([df1, df2], ignore_index=True)

### Instructor checkpoint

Ask students to predict the result of concatenating two Series with different indexes along `axis=1`. The important idea is that pandas aligns by **labels**, not simply by row position.


### Combining Data with Overlap

Sometimes two objects describe the same observations but one has missing values where the other has data. In that case, we may want to fill gaps rather than create additional rows or columns.

`combine_first()` keeps the calling object's non-missing values and fills its missing values from the other object after index/column alignment.


In [ ]:
a = pd.Series([np.nan, 2.5, np.nan, 3.5, 4.5, np.nan],
              index=['f', 'e', 'd', 'c', 'b', 'a'])
b = pd.Series(np.arange(len(a), dtype=np.float64),
              index=['f', 'e', 'd', 'c', 'b', 'a'])

# Label-based assignment is explicit and avoids ambiguous positional indexing.
b.loc['a'] = np.nan

display(a)
display(b)

# NumPy approach: choose b where a is missing; otherwise choose a.
pd.Series(np.where(a.isna(), b, a), index=a.index)


In [ ]:
b[:-2].combine_first(a[2:])

In [ ]:
df1 = pd.DataFrame({'a': [1., np.nan, 5., np.nan],
                    'b': [np.nan, 2., np.nan, 6.],
                    'c': range(2, 18, 4)})
df2 = pd.DataFrame({'a': [5., 4., np.nan, 3., 7.],
                    'b': [np.nan, 3., 4., 6., 8.]})

display(df1)
display(df2)
combined = df1.combine_first(df2)
combined


## 3. Reshaping and Pivoting

Wrangling often changes the **shape** of data without changing its underlying information.

- **Wide format:** one row per entity, many measurement columns.
- **Long format:** one row per entity-variable observation.

Long format is often convenient for grouped analysis and visualization; wide format is often convenient for human-readable reports and some modeling workflows.


### Reshaping with Hierarchical Indexing

In [ ]:
data = pd.DataFrame(np.arange(6).reshape((2, 3)),
                    index=pd.Index(['Ohio', 'Colorado'], name='state'),
                    columns=pd.Index(['one', 'two', 'three'],
                    name='number'))
data

In [ ]:
result = data.stack()
result

In [ ]:
result.unstack()

In [ ]:
result.unstack(0)
result.unstack('state')

In [ ]:
s1 = pd.Series([0, 1, 2, 3], index=['a', 'b', 'c', 'd'])
s2 = pd.Series([4, 5, 6], index=['c', 'd', 'e'])
data2 = pd.concat([s1, s2], keys=['one', 'two'])
data2
data2.unstack()

In [ ]:
wide = data2.unstack()
print('Wide form:')
display(wide)

print('Stacked back to long form (missing combinations dropped by default):')
display(wide.stack())

print('Preserving missing combinations:')
try:
    display(wide.stack(future_stack=True))
except TypeError:
    display(wide.stack(dropna=False))


In [ ]:
df = pd.DataFrame({'left': result, 'right': result + 5},
                  columns=pd.Index(['left', 'right'], name='side'))
df
df.unstack('state')

In [ ]:
df.unstack('state').stack('side')

### Instructor note: long versus wide data

A useful mental model is:

- `melt()` makes a dataset **longer and narrower**.
- `pivot()` / `unstack()` usually make a dataset **wider and shorter**.

Ask students to identify the observational unit before and after each reshape.


### Pivoting “Long” to “Wide” Format

In [ ]:
# Portable macroeconomic example.
# Prefer statsmodels' built-in dataset when available; otherwise use a small fallback dataset.
try:
    import statsmodels.api as sm
    data = sm.datasets.macrodata.load_pandas().data.copy()
except Exception:
    data = pd.DataFrame({
        'year': [2000, 2000, 2000, 2000, 2001, 2001, 2001, 2001],
        'quarter': [1, 2, 3, 4, 1, 2, 3, 4],
        'realgdp': [100, 101, 102, 103, 104, 105, 106, 107],
        'infl': [2.1, 2.2, 2.0, 2.3, 2.4, 2.2, 2.1, 2.0],
        'unemp': [4.0, 4.1, 4.2, 4.1, 4.0, 3.9, 3.8, 3.7]
    })

periods = pd.PeriodIndex(year=data['year'].astype(int),
                         quarter=data['quarter'].astype(int),
                         name='date')
columns = pd.Index(['realgdp', 'infl', 'unemp'], name='item')
data = data.reindex(columns=columns)
data.index = periods.to_timestamp(how='end').normalize()
ldata = data.stack().reset_index(name='value')


In [ ]:
ldata[:10]

In [ ]:
pivoted = ldata.pivot(index='date', columns='item', values='value')
pivoted

In [ ]:
ldata['value2'] = np.random.randn(len(ldata))
ldata[:10]

In [ ]:
# If we omit values=, pivot creates a hierarchical column index for every value column.
ldata['value2'] = np.random.randn(len(ldata))
pivoted_multi = ldata.pivot(index='date', columns='item')
display(pivoted_multi.head())

print('Selecting only the original value block:')
display(pivoted_multi['value'].head())


In [ ]:
unstacked = ldata.set_index(['date', 'item']).unstack('item')
unstacked[:7]

### Pivoting “Wide” to “Long” Format

In [ ]:
df = pd.DataFrame({'key': ['foo', 'bar', 'baz'],
                   'A': [1, 2, 3],
                   'B': [4, 5, 6],
                   'C': [7, 8, 9]})
df

In [ ]:
melted = pd.melt(df, ['key'])
melted

In [ ]:
reshaped = melted.pivot(index='key', columns='variable', values='value')
reshaped

In [ ]:
reshaped.reset_index()

In [ ]:
pd.melt(df, id_vars=['key'], value_vars=['A', 'B'])

In [ ]:
print('Melt only selected numeric columns:')
display(pd.melt(df, value_vars=['A', 'B', 'C']))

print('Including key as a measured variable changes its role and usually is not what we want:')
display(pd.melt(df, value_vars=['key', 'A', 'B']))


In [ ]:
# Restore the previous directory only if %pushd was used successfully.
if course_dir.exists():
    %popd


## Conclusion and Validation Checklist

The key skill in data wrangling is not memorizing functions—it is choosing the operation that matches the structure of the problem.

| Goal | Main pandas tool |
|---|---|
| Match records by one or more keys | `pd.merge()` |
| Align objects by index | `.join()` |
| Stack objects by rows or columns | `pd.concat()` |
| Fill missing values from overlapping data | `.combine_first()` |
| Move index levels into columns | `.unstack()` |
| Move columns into index levels | `.stack()` |
| Convert tidy/long data to wide data | `.pivot()` |
| Convert wide data to long data | `pd.melt()` |

### Validate every wrangling step

After a merge, concat, or reshape, ask:

1. **Shape:** Did the number of rows and columns change as expected?
2. **Keys:** Are supposed-to-be-unique keys still unique?
3. **Missingness:** Did the operation create unexpected `NaN` values?
4. **Coverage:** Which records matched and which did not?
5. **Meaning:** Does one row still represent what you think it represents?

### Suggested in-class challenge

Create two small DataFrames: one with students and majors, and one with majors and department chairs. Merge them so every student remains in the result. Then use `indicator=True` to identify any student whose major did not match a department record.
